In [ ]:
from pathlib import Path
import json
import re
import textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from IPython.display import display
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'configs/proposal_mmd_v2.json').exists()), None)
if ROOT is None:
    raise RuntimeError('Mở notebook từ thư mục KLTN hoặc một thư mục con.')
CANONICAL = ROOT / 'results/canonical/tableau'
ANALYSIS = ROOT / 'results/analysis'
OUT = ANALYSIS / 'notebook_figures'
OUT.mkdir(parents=True, exist_ok=True)
SEED = 42
EXPORT = True
FIG_WIDTH_CM = 16.0
FIG_WIDTH = FIG_WIDTH_CM / 2.54
EXPORT_DPI = 300
METHOD_COLORS = {'source_only': '#64748B', 'marginal_mmd': '#2563EB', 'mk_mmd': '#F97316', 'class_aware_mmd': '#14B8A6'}
ROLE_CLASS_COLORS = {('Source', 'Benign'): '#2563EB', ('Source', 'Attack'): '#F97316', ('Target', 'Benign'): '#14B8A6', ('Target', 'Attack'): '#A855F7'}
DOMAIN_COLORS = {'UNSW-NB15': '#2563EB', 'CICIDS2017': '#14B8A6', 'unsw': '#2563EB', 'cicids': '#14B8A6'}
CLASS_COLORS = {'benign': '#2563EB', 'attack': '#F97316'}
DIRECTION_COLORS = {'unsw_to_cicids': '#2563EB', 'cicids_to_unsw': '#14B8A6'}
PHASE_COLORS = {'before': '#64748B', 'after': '#2563EB'}
FEATURE_COLORS = dict(zip(['flow_duration', 'fwd_packets', 'bwd_packets', 'fwd_bytes', 'bwd_bytes'], ['#64748B', '#2563EB', '#F97316', '#14B8A6', '#A855F7']))
EVALUATION_COLORS = {'within': '#2563EB', 'cross': '#F97316', 'Within-domain': '#2563EB', 'Cross-domain': '#F97316'}
COLORS = list(METHOD_COLORS.values())
FIGURE_METADATA = {}

def semantic_color(values):
    values = values if isinstance(values, (tuple, list)) else [values]
    if len(values) == 2 and tuple(values) in ROLE_CLASS_COLORS:
        return ROLE_CLASS_COLORS[tuple(values)]
    for mapping in [METHOD_COLORS, DOMAIN_COLORS, CLASS_COLORS, PHASE_COLORS, FEATURE_COLORS, EVALUATION_COLORS]:
        for v in values:
            if v in mapping:
                return mapping[v]
    for (role, cls), color in ROLE_CLASS_COLORS.items():
        if f'{role} {cls}' in values:
            return color
    return '#64748B'

def bar_style(ax, bars, rows):
    for bar, (_, row) in zip(bars, rows.iterrows()):
        phase = row.get('phase')
        cls = str(row.get('class_group', row.get('class', ''))).lower()
        evaluation = row.get('evaluation')
        if phase == 'before' or cls == 'attack' or evaluation in ['cross', 'Cross-domain']:
            bar.set_hatch('//')
        bar.set_edgecolor('#FFFFFF')
        bar.set_linewidth(0.5)
METHODS = {'source_only': 'Source-only', 'marginal_mmd': 'MMD', 'mk_mmd': 'MK-MMD', 'class_aware_mmd': 'Class-Aware MMD'}
DIRECTIONS = {'unsw_to_cicids': 'UNSW-NB15 → CICIDS2017', 'cicids_to_unsw': 'CICIDS2017 → UNSW-NB15'}
LABELS = {'pr_auc': 'Average precision (AP)', 'ap': 'Average precision (AP)', 'mmd2': 'MMD²', 'domain_auc': 'Domain classifier AUC', 'adaptation_gain_pr_auc': 'Paired ΔAP', 'ks_statistic': 'KS statistic'}
plt.rcParams.update({'figure.facecolor': 'white', 'axes.facecolor': '#F8FAFC', 'axes.spines.top': False, 'axes.spines.right': False, 'axes.edgecolor': '#CBD5E1', 'axes.labelcolor': '#334155', 'text.color': '#0F172A', 'font.family': 'DejaVu Sans', 'font.size': 9, 'axes.titlesize': 10, 'axes.titleweight': 'bold', 'axes.labelsize': 9, 'xtick.labelsize': 8, 'ytick.labelsize': 8, 'legend.fontsize': 8, 'pdf.fonttype': 42, 'ps.fonttype': 42, 'axes.grid': True, 'grid.color': '#E2E8F0', 'grid.alpha': 0.7, 'axes.axisbelow': True, 'axes.prop_cycle': plt.cycler(color=COLORS), 'savefig.facecolor': 'white'})

def label(x):
    return METHODS.get(x, DIRECTIONS.get(x, LABELS.get(x, str(x).replace('_', ' ').capitalize())))

def load(path, *, filters=None, columns=None, chunksize=100000):
    """Filters are explicit and identical for every file size; chunking only controls memory."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f'CSV cần thiết chưa có: {path}')
    filters = filters or {}
    available = pd.read_csv(path, nrows=0).columns.tolist()
    missing = set(filters) - set(available)
    if missing:
        raise ValueError(f'{path.name}: missing filter columns {sorted(missing)}')
    if columns is not None and (not set(filters).issubset(columns)):
        raise ValueError('Selected columns must include filter columns')
    parts = []
    for chunk in pd.read_csv(path, usecols=columns, chunksize=chunksize):
        for key, value in filters.items():
            chunk = chunk[chunk[key].isin(value if isinstance(value, (list, tuple, set)) else [value])]
        if len(chunk):
            parts.append(chunk)
    result = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(columns=columns or available)
    if filters and result.empty:
        raise ValueError(f'{path.name}: no rows for explicit filters {filters}')
    result.attrs['source_csv'] = str(path.relative_to(ROOT))
    result.attrs['filters'] = filters
    return result

def fit_category_labels(ax):
    """Use horizontal category labels when adjacent text fits; otherwise rotate 90°."""
    if ax.xaxis.get_major_formatter().__class__.__name__ == 'ScalarFormatter':
        return
    ticks = [tick for tick in ax.get_xticklabels() if tick.get_visible() and tick.get_text()]
    if not ticks or not any(any(ch.isalpha() for ch in tick.get_text()) for tick in ticks):
        return
    renderer = ax.figure.canvas.get_renderer()
    extents=[]
    for tick in ticks:
        width=max(renderer.get_text_width_height_descent(line, tick.get_fontproperties(), False)[0]
                  for line in tick.get_text().splitlines())
        center=ax.transData.transform((tick.get_position()[0],0))[0]
        extents.append((center-width/2,center+width/2))
    fits=all(right+8 <= next_left for (_,right),(next_left,_) in zip(extents,extents[1:]))
    for tick in ticks:
        tick.set_rotation(0 if fits else 90)
        tick.set_ha('center'); tick.set_va('top')


def finish(fig, title, note=None, large=False):
    width, height = fig.get_size_inches()
    large = large or len([ax for ax in fig.axes if ax.get_visible()]) >= 4
    target_width = 26 / 2.54 if large else FIG_WIDTH
    fig.set_size_inches(target_width, min(24 / 2.54, max(4.5, height * target_width / width)))
    for ax in fig.axes:
        ax.set_title(textwrap.fill(ax.get_title(), width=max(22, int(ax.get_position().width * target_width * 9))), fontsize=11 if large else 10)
        ax.set_ylabel(textwrap.fill(ax.get_ylabel(), width=38), fontsize=9)
    if large:
        for i, ax in enumerate((ax for ax in fig.axes if ax.get_visible())):
            ax.text(-0.03, 1.12, f'({chr(97 + i)})', transform=ax.transAxes, fontsize=12, fontweight='bold', ha='left', va='bottom')
            ax.tick_params(labelsize=9)
            ax.xaxis.label.set_size(10)
            ax.yaxis.label.set_size(10)
    heading = textwrap.fill(title, width=100 if large else 68)
    footer = textwrap.fill(note, width=95) if note else None
    fig.suptitle(heading, x=0.02, y=0.99, ha='left', fontsize=14 if large else 11, fontweight='bold')
    if footer:
        fig.text(0.02, 0.01, footer, fontsize=7, color='#64748B', va='bottom')
    top = 1 - (0.2 * len(heading.splitlines()) + 0.2) / fig.get_figheight()
    fig.tight_layout(rect=[0.01, 0.08, 0.99, top], h_pad=1.8, w_pad=1.2)
    for ax in fig.axes:
        fit_category_labels(ax)
    fig.tight_layout(rect=[0.01, 0.08, 0.99, top], h_pad=1.8, w_pad=1.2)
    if EXPORT:
        name = re.sub('[^a-zA-Z0-9_-]+', '_', title).strip('_').lower()
        FIGURE_METADATA[name] = {'title': title, 'data_role_and_uncertainty': note}
        fig.savefig(OUT / f'{name}.png', dpi=EXPORT_DPI)
    plt.show()
    plt.close(fig)
paths = sorted((ROOT / 'results').rglob('*.csv'))
display(pd.DataFrame([{'CSV': str(p.relative_to(ROOT)), 'MiB': round(p.stat().st_size / 2 ** 20, 2)} for p in paths]))

## 01 · Hiệu năng trên target validation
Mean và CI 95% lấy trực tiếp từ summary CSV. Paired ΔAP được exporter tính so với source-only cùng seed. Không lấy trung bình các phase hoặc direction với nhau.

In [ ]:
def summary_plot(
    df,
    title,
    group_cols=("method",),
    value="mean",
    low="ci95_low",
    high="ci95_high",
    facet="direction",
    ylabel=None,
    note=None,
):
    facets = list(df[facet].dropna().unique()) if facet and facet in df else ["All"]
    fig, axes = plt.subplots(
        len(facets), 1, figsize=(FIG_WIDTH, 4.8 * len(facets)), squeeze=False
    )
    for ax, f in zip(axes.flat, facets):
        d = df[df[facet] == f].copy() if facet and facet in df else df.copy()
        d = d.reset_index(drop=True)
        names = [
            " · ".join((label(row[c]) for c in group_cols)) for _, row in d.iterrows()
        ]
        x = np.arange(len(d))
        y = d[value].to_numpy(float)
        bars = ax.bar(
            x,
            y,
            color=[
                semantic_color([row[c] for c in group_cols]) for _, row in d.iterrows()
            ],
            width=0.65,
        )
        bar_style(ax, bars, d)
        if low in d and high in d:
            good = d[[value, low, high]].notna().all(axis=1).to_numpy()
            ax.errorbar(
                x[good],
                y[good],
                yerr=np.maximum(
                    0,
                    np.vstack(
                        [y[good] - d.loc[good, low], d.loc[good, high] - y[good]]
                    ),
                ),
                fmt="none",
                ecolor="#334155",
                capsize=4,
            )
        ax.set_xticks(x, names, rotation=90, ha="center")
        ax.set_title(label(f))
        ax.set_ylabel(ylabel or label(value))
        ax.axhline(0, color="#94A3B8", lw=0.8)
    finish(fig, title, note)


def target_ap_grouped_bar(performance):
    """Mean ± sample SD, paired columns for the two transfer directions."""
    methods = ['source_only', 'marginal_mmd', 'mk_mmd', 'class_aware_mmd']
    directions = ['unsw_to_cicids', 'cicids_to_unsw']
    data = performance[performance.metric == 'pr_auc'].copy()
    if data.duplicated(['direction', 'method']).any():
        raise ValueError('Duplicate AP summary rows')
    indexed = data.set_index(['direction', 'method'])
    fig, ax = plt.subplots(figsize=(FIG_WIDTH, 4.6))
    x = np.arange(len(methods)); width = .34
    for i, direction in enumerate(directions):
        rows = indexed.loc[[(direction, method) for method in methods]]
        if not (rows.n_seeds == 5).all() or rows[['mean', 'std']].isna().any().any():
            raise ValueError('AP grouped bar requires five-seed mean and sample SD')
        ax.bar(x + (i - .5) * width, rows['mean'], width=width,
               yerr=rows['std'], capsize=4, label=DIRECTIONS[direction],
               color=DIRECTION_COLORS[direction],
               error_kw={'ecolor': '#334155', 'elinewidth': 1.1},
               edgecolor='white', linewidth=.6)
    ax.set_xticks(x, ['Source-only', 'Marginal\nMMD', 'MK-MMD', 'Class-Aware\nMMD'],
                  rotation=0, ha='center')
    ax.set_ylabel('Target-domain average precision (AP)')
    ax.set_ylim(bottom=0)
    ax.set_title('Target-domain Average Precision by Method', fontsize=11, pad=14)
    ax.xaxis.grid(False)
    ax.legend(loc='upper center', bbox_to_anchor=(.5, -.24), ncol=2,
              frameon=False, fontsize=8, columnspacing=1.5)
    fig.text(.5, .01, 'Target validation · five training seeds · mean ± sample SD',
             ha='center', fontsize=8, color='#64748B')
    fig.tight_layout(rect=[0, .04, 1, 1])
    fit_category_labels(ax)
    fig.tight_layout(rect=[0, .04, 1, 1])
    if EXPORT:
        fig.savefig(OUT / '01_target_validation_average_precision_ap.png', dpi=EXPORT_DPI)
    plt.show()
    plt.close(fig)


def paired_gain_point_plot(performance):
    """Paired Student-t 95% CIs for adaptation minus source-only AP."""
    methods=['marginal_mmd','mk_mmd','class_aware_mmd']
    directions=['unsw_to_cicids','cicids_to_unsw']
    gain=performance[(performance.metric=='adaptation_gain_pr_auc') &
                     performance.method.isin(methods)].copy()
    if gain.duplicated(['direction','method']).any():
        raise ValueError('Duplicate paired-gain summary rows')
    indexed=gain.set_index(['direction','method'])
    fig,ax=plt.subplots(figsize=(FIG_WIDTH,4.8))
    y=np.arange(len(methods))
    for i,direction in enumerate(directions):
        rows=indexed.loc[[(direction,method) for method in methods]]
        if not (rows.n_seeds==5).all() or rows[['mean','ci95_low','ci95_high']].isna().any().any():
            raise ValueError('Paired gain requires five-seed means and exported 95% CIs')
        x=rows['mean'].to_numpy()
        errors=np.vstack([x-rows.ci95_low.to_numpy(),rows.ci95_high.to_numpy()-x])
        if np.any(errors<0): raise ValueError('Invalid paired confidence interval')
        ax.errorbar(x,y+(i-.5)*.22,xerr=errors,
                    fmt='o' if i==0 else 's',markersize=6,
                    color=DIRECTION_COLORS[direction],elinewidth=1.5,capsize=4,
                    label=DIRECTIONS[direction])
    ax.axvline(0,color='#475569',ls='--',lw=1.2,zorder=0)
    ax.set_yticks(y,['Marginal MMD','MK-MMD','Class-Aware MMD'])
    ax.invert_yaxis()
    ax.set_ylim(len(methods)-.55,-.45)
    ax.set_xlabel('Paired ΔAP (adapted − source-only)')
    ax.set_title('Paired Adaptation Gain (ΔAP)',fontsize=11,pad=14)
    ax.yaxis.grid(False)
    ax.set_xlim(min(gain.ci95_low.min(),0)-.025,max(gain.ci95_high.max(),0)+.025)
    ax.legend(loc='upper center',bbox_to_anchor=(.5,-.20),
              ncol=1,frameon=False,fontsize=8)
    fig.text(.5,.012,'Target validation · five paired training seeds · 95% Student-t CI',
             ha='center',fontsize=7.5,color='#64748B')
    fig.tight_layout(rect=[0,.05,1,1])
    if EXPORT:
        fig.savefig(OUT/'01_target_validation_paired_ap.png',dpi=EXPORT_DPI)
    plt.show()
    plt.close(fig)


performance = load(CANONICAL / 'performance_summary.csv')
for metric in performance.metric.unique():
    if metric == 'pr_auc':
        target_ap_grouped_bar(performance)
    elif metric == 'adaptation_gain_pr_auc':
        paired_gain_point_plot(performance)
    else:
        summary_plot(performance[performance.metric == metric],
                     f'01 · Target validation — {label(metric)}',
                     ylabel=label(metric),
                     note='Target validation · mean and exported 95% CI across five training seeds')


### Confusion matrices và độ biến thiên paired gain
Confusion matrix hiển thị trung bình counts giữa seeds; phần trăm chuẩn hóa theo true class. Điểm ΔAP giữ từng seed.

In [ ]:
runs = load(CANONICAL / "performance_runs.csv")
for direction, d in runs.groupby("direction"):
    groups = list(d.groupby("method"))
    fig, axes = plt.subplots(
        int(np.ceil(len(groups) / 2)),
        2,
        figsize=(FIG_WIDTH, 3.8 * np.ceil(len(groups) / 2)),
        squeeze=False,
    )
    for ax, (method, g) in zip(axes.flat, groups):
        counts = (
            g[
                [
                    "confusion_matrix_tn",
                    "confusion_matrix_fp",
                    "confusion_matrix_fn",
                    "confusion_matrix_tp",
                ]
            ]
            .mean()
            .to_numpy()
            .reshape(2, 2)
        )
        rates = counts / counts.sum(axis=1, keepdims=True)
        ax.imshow(rates, cmap="Blues", vmin=0, vmax=1)
        ax.grid(False)
        for i in range(2):
            for j in range(2):
                ax.text(
                    j,
                    i,
                    f"{rates[i, j]:.1%}\n{counts[i, j]:,.0f} mean",
                    ha="center",
                    va="center",
                    color="white" if rates[i, j] > 0.5 else "#0F172A",
                )
        ax.set_xticks([0, 1], ["Benign", "Attack"])
        ax.set_yticks([0, 1], ["Benign", "Attack"])
        ax.set_xlabel("Predicted class")
        ax.set_ylabel("True class")
        ax.set_title(label(method))
    finish(
        fig,
        f"01 · Confusion matrices — {direction}",
        "Target validation · mean counts over seeds · row-normalized percentages",
    )
fig, axes = plt.subplots(2, 1, figsize=(FIG_WIDTH, 8.6))
for ax, (direction, d) in zip(axes, runs.groupby("direction")):
    methods = list(d.method.unique())
    for i, method in enumerate(methods):
        g = d[d.method == method].sort_values("seed")
        ax.scatter(
            i + np.linspace(-0.12, 0.12, len(g)),
            g.adaptation_gain_pr_auc,
            s=50,
            color=METHOD_COLORS[method],
            alpha=0.85,
        )
    ax.set_xticks(
        range(len(methods)), [label(m) for m in methods], rotation=90, ha="center"
    )
    ax.axhline(0, ls="--", color="#64748B")
    ax.set_ylabel("Paired ΔAP vs source-only")
    ax.set_title(label(direction))
finish(
    fig,
    "01 · Adaptation gain by individual seed",
    "Target validation · paired ΔAP per individual training seed",
)

## 02 · Within-domain và cross-domain AP / RQ1
Within-domain trong bảng AP có thể đánh giá trên miền nguồn của hướng chuyển; đọc cùng prevalence và nguồn CSV. RQ1 so sánh within/cross trên cùng miền đánh giá, được trình bày riêng.

In [ ]:
def rq1_ap_grouped_bar(rq):
    """Within/cross AP on the same evaluation split; mean ± sample SD."""
    domains = ['cicids', 'unsw']
    if rq.eval_domain.duplicated().any():
        raise ValueError('Duplicate RQ1 evaluation domains')
    rows = rq.set_index('eval_domain').loc[domains]
    if not (rows.n_seeds == 5).all():
        raise ValueError('RQ1 AP chart requires five training seeds')
    fig, ax = plt.subplots(figsize=(FIG_WIDTH, 4.5))
    x = np.arange(len(domains)); width = .32
    for i, mode in enumerate(['within','cross']):
        mean=rows[f'{mode}_pr_auc_mean']
        sd=rows[f'{mode}_pr_auc_std']
        if mean.isna().any() or sd.isna().any():
            raise ValueError('Missing RQ1 AP mean or SD')
        ax.bar(x + (i-.5)*width, mean, width=width, yerr=sd,
               capsize=4, color=EVALUATION_COLORS[mode],
               label='Within-domain' if mode=='within' else 'Cross-domain',
               edgecolor='white', linewidth=.6,
               error_kw={'ecolor':'#334155','elinewidth':1.1})
    ax.set_xticks(x,['CICIDS2017','UNSW-NB15'],rotation=0,ha='center')
    ax.set_xlabel('Evaluation domain (same validation split for each comparison)')
    ax.set_ylabel('Average precision (AP)')
    ax.set_ylim(0,1.05)
    ax.set_title('Within-Domain vs Cross-Domain Average Precision',fontsize=11,pad=14)
    ax.xaxis.grid(False)
    ax.legend(loc='upper center',bbox_to_anchor=(.5,-.22),ncol=2,frameon=False,fontsize=9)
    fig.text(.5,.012,'Development validation · five training seeds · mean ± sample SD',
             ha='center',fontsize=7.5,color='#64748B')
    fig.tight_layout(rect=[0,.05,1,1])
    if EXPORT:
        fig.savefig(OUT/'02_rq1_average_precision_ap.png',dpi=EXPORT_DPI)
    plt.show()
    plt.close(fig)


within = load(ANALYSIS / "tableau_ap_pr/within_cross_ap_summary.csv")
summary_plot(
    within,
    "02 · Within-domain vs cross-domain AP",
    ("method", "evaluation"),
    value="ap_mean",
    ylabel="Average precision (AP)",
    note="Development validation · five training seeds · CI only where present in summary",
)
rq = load(ANALYSIS / "rq1/rq1_summary.csv")
for metric in ["pr_auc", "roc_auc", "macro_f1", "f1_attack", "recall", "fpr"]:
    rows = []
    for _, r in rq.iterrows():
        for mode in ["within", "cross"]:
            rows.append(
                {
                    "eval_domain": r.eval_domain,
                    "evaluation": mode,
                    "mean": r[f"{mode}_{metric}_mean"],
                }
            )
    if metric == 'pr_auc':
        rq1_ap_grouped_bar(rq)
    else:
        summary_plot(
            pd.DataFrame(rows),
            f"02 · RQ1 — {label(metric)}",
            ("evaluation",),
            facet="eval_domain",
            ylabel=label(metric),
            note="Development validation · five training seeds · CI only where present in summary",
        )
    d = rq.rename(
        columns={
            f"drop_{metric}_mean": "mean",
            f"drop_{metric}_ci95_low": "ci95_low",
            f"drop_{metric}_ci95_high": "ci95_high",
        }
    )
    summary_plot(
        d,
        f"02 · RQ1 within minus cross — {label(metric)}",
        ("eval_domain",),
        facet=None,
        ylabel="Within − cross",
        note="Development validation · five training seeds · CI only where present in summary",
    )

## 03 · Precision–Recall curves
Vẽ đúng thứ tự điểm trong CSV. Seed được chọn ở phần cấu hình. Đường ngang là attack prevalence khi CSV cung cấp; không gộp PR curves giữa seeds.

In [ ]:
def curves(
    df, x, y, groups, facets, title, order=None, xlabel=None, ylabel=None, note=None
):
    facet_cols = list(facets)
    grouped = (
        list(df.groupby(facet_cols, sort=False, dropna=False))
        if facet_cols
        else [("All", df)]
    )
    cols = min(2, len(grouped))
    rows = int(np.ceil(len(grouped) / cols))
    fig, axes = plt.subplots(
        rows, cols, figsize=(FIG_WIDTH, 3.0 * rows + 1.0), squeeze=False
    )
    for ax, (key, d) in zip(axes.flat, grouped):
        for series, g in d.groupby(list(groups), sort=False, dropna=False):
            if order:
                g = g.sort_values(order)
            name = " · ".join(
                (label(v) for v in (series if isinstance(series, tuple) else (series,)))
            )
            ax.plot(
                g[x],
                g[y],
                label=name,
                color=semantic_color(series if isinstance(series, tuple) else [series]),
                linestyle=(
                    "--"
                    if "before" in (series if isinstance(series, tuple) else [series])
                    else "-"
                ),
                lw=1.7,
            )
        if x == "quantile_order" and "quantile" in d:
            ticks = d[[x, "quantile"]].drop_duplicates().sort_values(x)
            ax.set_xticks(ticks[x], ticks["quantile"].str.upper())
        if x == "recall" and "prevalence" in d:
            ax.axhline(
                d.prevalence.iloc[0],
                ls="--",
                color="#64748B",
                label="Attack prevalence",
            )
        ax.set_title(
            " · ".join((label(v) for v in (key if isinstance(key, tuple) else (key,))))
        )
        ax.set_xlabel(xlabel or label(x))
        ax.set_ylabel(ylabel or label(y))
        ax.legend(fontsize=8)
    for ax in list(axes.flat)[len(grouped) :]:
        ax.set_visible(False)
    finish(fig, title, note)


pr = load(
    ANALYSIS / "tableau_ap_pr/target_pr_curves_seed42.csv",
    filters={"training_seed": SEED},
)
curves(
    pr[pr.training_seed == SEED],
    "recall",
    "precision",
    ["method"],
    ["direction"],
    f"03 · Target PR curves — seed {SEED}",
    order="point_order",
    note=f"Target validation · training seed {SEED} · diagnostic replay",
)
pr_before = load(CANONICAL / "pr_curves.csv", filters={"seed": SEED})
curves(
    pr_before[pr_before.seed == SEED],
    "recall",
    "precision",
    ["phase"],
    ["direction", "method"],
    f"03 · PR before and after adaptation — seed {SEED}",
    order="point_index",
    note=f"Target validation · training seed {SEED} · diagnostic replay",
)

## 04 · Feature shift — KS và ECDF
Ưu tiên snapshot 5 sampling seeds. ECDF hiển thị một sampling seed, không trộn các phân phối. Trục signed log1p dùng giá trị đã xuất trong CSV.

In [ ]:
shift = ANALYSIS / 'tableau_feature_shift_5seeds'
ks = load(shift / 'ks_summary_common_features.csv')
summary_plot(ks, '04 · Common-feature domain shift — KS', ('feature',), value='ks_mean', facet=None, ylabel='Mean KS statistic', note='Unlabeled training samples · intrinsic diagnostics · five sampling seeds')
ks_runs = load(shift / 'ks_ranking_common_features.csv')
fig, ax = plt.subplots(figsize=(FIG_WIDTH, 5.2))
features = ks.feature.tolist()
for i, f in enumerate(features):
    vals = ks_runs.loc[ks_runs.feature == f, 'ks_statistic']
    ax.scatter(np.full(len(vals), i), vals, color=FEATURE_COLORS[f], s=45)
ax.set_xticks(range(len(features)), [label(f) for f in features], rotation=90)
ax.set_ylabel('KS statistic')
finish(fig, '04 · KS variability across sampling seeds', 'Unlabeled training samples · KS per sampling seed')
ecdf = load(shift / 'ecdf_common_features.csv', filters={'sampling_seed': SEED})
curves(ecdf[ecdf.sampling_seed == SEED], 'value_log1p', 'ecdf', ['domain'], ['feature'], f'04 · Common-feature ECDF — sampling seed {SEED}', order='point_order', xlabel='Signed log1p(raw feature)', ylabel='Cumulative probability', note=f'Unlabeled training samples · sampling seed {SEED} · intrinsic ECDF')
classks = load(ANALYSIS / 'tableau_class_diagnostics/class_conditional_ks.csv')
summary_plot(classks, '04 · Class-conditional domain shift — KS', ('feature',), value='ks_statistic', facet='class', ylabel='KS statistic', note='Labeled validation samples · post-hoc · class-conditional KS')
class_root = ANALYSIS / 'tableau_class_diagnostics'
class_manifest = json.loads((class_root / 'manifest.json').read_text())
ce_path = class_root / 'class_conditional_ecdf.csv'
header = pd.read_csv(ce_path, nrows=0).columns
ce_filters = {'sampling_seed': SEED} if 'sampling_seed' in header else {}
ce = load(ce_path, filters=ce_filters)
assert class_manifest['data_role'] == 'post_hoc_validation_diagnostic_only'
assert class_manifest['source_split'] == 'unsw_val' and class_manifest['target_split'] == 'cicids_val'
assert np.allclose(ce.value_log1p, np.sign(ce.value_raw) * np.log1p(np.abs(ce.value_raw)), rtol=1e-10, atol=1e-12)
for (domain, role, cls, feature), part in ce.groupby(['domain', 'role', 'class', 'feature']):
    key = {'UNSW-NB15': 'unsw', 'CICIDS2017': 'cicids'}[domain]
    assert role == ('Source' if key == 'unsw' else 'Target')
    count = class_manifest['datasets'][key]['class_counts']['0' if cls == 'Benign' else '1']
    assert part.sample_n.nunique() == 1 and part.sample_n.iloc[0] == count
    ordered = part.sort_values('point_order')
    assert np.all(np.diff(ordered.ecdf) >= -1e-12) and np.isclose(ordered.ecdf.iloc[-1], 1)
features = ['flow_duration', 'fwd_packets', 'bwd_packets', 'fwd_bytes', 'bwd_bytes']
assert set(ce.feature.unique()) == set(features)
palette = {f'{role} {cls}': color for (role, cls), color in ROLE_CLASS_COLORS.items()}
fig, axes = plt.subplots(3, 2, figsize=(16 / 2.54, 20 / 2.54))
for i, (ax, feature) in enumerate(zip(axes.flat, features)):
    for series, color in palette.items():
        part = ce[(ce.feature == feature) & (ce.series == series)].sort_values('point_order')
        if part.empty:
            raise ValueError(f'Missing ECDF series: {feature}/{series}')
        ax.plot(part.value_log1p, part.ecdf, color=color, lw=1.4)
    ax.set_title(f'({chr(97 + i)}) {label(feature)}', fontsize=9)
    ax.set_ylim(0, 1.02)
    ax.set_xlabel('Signed log1p(feature value)', fontsize=8)
    ax.set_ylabel('ECDF', fontsize=8)
    ax.tick_params(labelsize=7.5)
axes[2, 1].axis('off')
handles = [Line2D([0], [0], color=color, lw=2, label=series) for series, color in palette.items()]
axes[2, 1].legend(handles=handles, loc='center', frameon=False, fontsize=9)
fig.tight_layout(pad=1.2)
figure_name = 'figure_4_1_class_ecdf' + (f'_sampling_seed{SEED}' if ce_filters else '')
caption = 'Hình 4.1. Phân phối ECDF theo lớp của năm đặc trưng chung trên validation của UNSW-NB15 (Source) và CICIDS2017 (Target), hướng UNSW → CICIDS. Dữ liệu complete cases từ unsw_val và cicids_val; phân tích hậu nghiệm có nhãn. Trục hoành dùng signed log1p; không phải dữ liệu huấn luyện thích nghi. ' + (f'Sampling seed {SEED}.' if ce_filters else 'ECDF dùng toàn bộ complete cases, không áp dụng sampling seed; seed 42 trong manifest chỉ dành cho scatter.')
metadata = {'source_csv': str(ce_path.relative_to(ROOT)), 'manifest': class_manifest, 'filters': ce_filters, 'transform_verified': 'sign(x)*log1p(abs(x)) over all CSV rows', 'sample_counts_verified_against_manifest': True, 'caption': caption}
if EXPORT:
    fig.savefig(OUT / f'{figure_name}.png', dpi=300, bbox_inches='tight')
print(caption)
plt.show()
plt.close(fig)

## 05 · Joint features và latent UMAP
Joint feature scatter dùng mẫu CSV nguyên trạng, tách bản balanced. UMAP lấy tọa độ có sẵn; khoảng cách giữa các panel embedding khác nhau không được coi là thước đo MMD. Class diagnostics không có cột direction được giữ theo Source/Target như exporter, không nhân sang hướng ngược lại.

In [ ]:
for filename in ["joint_feature_validation", "joint_feature_validation_balanced"]:
    joint = load(ANALYSIS / f"tableau_class_diagnostics/{filename}.csv")
    curves_data = joint.copy()
    pairs = [
        ("fwd_packets_log1p", "bwd_packets_log1p"),
        ("fwd_bytes_log1p", "bwd_bytes_log1p"),
        ("flow_duration_log1p", "fwd_bytes_log1p"),
    ]
    fig, axes = plt.subplots(3, 1, figsize=(FIG_WIDTH, 12))
    for ax, (x, y) in zip(axes, pairs):
        for i, ((role, cls), g) in enumerate(joint.groupby(["role", "class"])):
            ax.scatter(
                g[x],
                g[y],
                s=8,
                alpha=0.25,
                color=ROLE_CLASS_COLORS[role, cls],
                label=f"{role} {cls}",
                rasterized=True,
            )
        ax.set_xlabel(label(x))
        ax.set_ylabel(label(y))
        ax.legend(markerscale=2, fontsize=8)
    finish(
        fig,
        f"05 · Joint feature distributions — {filename.replace('joint_feature_validation', 'validation')}",
        "Validation samples · signed log1p axes · sample counts are not dataset totals",
    )
umap = load(
    ANALYSIS / "tableau_class_diagnostics/latent_umap_seed42/latent_umap.csv",
    filters={"training_seed": SEED, "sampling_seed": SEED},
)
u = umap[(umap.training_seed == SEED) & (umap.sampling_seed == SEED)]
for direction, d in u.groupby("direction"):
    models = list(d.model.unique())
    fig, axes = plt.subplots(
        int(np.ceil(len(models) / 2)),
        2,
        figsize=(FIG_WIDTH, 3.4 * np.ceil(len(models) / 2) + 0.8),
        squeeze=False,
    )
    for ax, model in zip(axes.flat, models):
        for i, ((role, cls), g) in enumerate(
            d[d.model == model].groupby(["role", "class"])
        ):
            ax.scatter(
                g.umap_1,
                g.umap_2,
                s=7,
                alpha=0.4,
                color=ROLE_CLASS_COLORS[role, cls],
                label=f"{role} {cls}",
                rasterized=True,
            )
        ax.set_title(model)
        ax.set_xlabel("UMAP 1")
        ax.set_ylabel("UMAP 2")
        ax.legend(fontsize=8)
    finish(
        fig,
        f"05 · Latent UMAP — {direction} — seed {SEED}",
        f"Labeled validation samples · training and sampling seed {SEED} · existing embedding, post-hoc",
    )

## 06 · Alignment, negative transfer và prediction diagnostics
Before/after và từng class được giữ riêng. Mean±SD của raw runs là độ phân tán giữa training seeds, không phải CI. Quantile profile không phải density plot.

In [ ]:
def run_plot(df, metrics, title, extra=()):
    for metric in metrics:
        keys = [c for c in ["direction", "method", *extra] if c in df]
        d = df.groupby(keys, dropna=False)[metric].agg(["mean", "std"]).reset_index()
        d["sd_low"] = d["mean"] - d["std"]
        d["sd_high"] = d["mean"] + d["std"]
        role = (
            "Post-hoc gradient diagnostic · source_train labels and target_train features"
            if metric.startswith("gradient_conflict")
            else "Post-hoc validation diagnostic"
        )
        summary_plot(
            d,
            f"{title} — {label(metric)}",
            tuple((c for c in keys if c != "direction")),
            ylabel=f"{label(metric)} (mean ± SD)",
            low="sd_low",
            high="sd_high",
            note=f"{role} · mean ± SD across five training seeds",
        )


def latent_mmd_grouped_bar(alignment):
    methods = ['marginal_mmd', 'mk_mmd', 'class_aware_mmd']
    directions = ['unsw_to_cicids', 'cicids_to_unsw']
    if alignment.duplicated(['direction', 'method', 'phase', 'seed']).any():
        raise ValueError('Duplicate alignment runs')
    stats = alignment.groupby(['direction', 'method', 'phase']).mmd2.agg(['mean', 'std', 'count'])
    fig, axes = plt.subplots(2, 1, figsize=(FIG_WIDTH, 17/2.54), sharey=True)
    x = np.arange(len(methods)); width = .34
    for ax, direction in zip(axes, directions):
        for i, phase in enumerate(['before', 'after']):
            rows = stats.loc[[(direction, method, phase) for method in methods]]
            if not (rows['count'] == 5).all() or rows[['mean', 'std']].isna().any().any():
                raise ValueError('MMD figure requires five runs in every group')
            ax.bar(x + (i-.5)*width, rows['mean'], width=width,
                   yerr=rows['std'], capsize=4, label=phase.title(),
                   color=PHASE_COLORS[phase], edgecolor='white', linewidth=.6,
                   error_kw={'ecolor':'#334155', 'elinewidth':1.1})
        ax.set_xticks(x, ['Marginal MMD','MK-MMD','Class-Aware MMD'],rotation=0,ha='center')
        ax.set_ylabel('Latent-space MMD²')
        ax.set_title(DIRECTIONS[direction],fontsize=10)
        ax.set_ylim(bottom=0)
        ax.xaxis.grid(False)
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc='lower center', bbox_to_anchor=(.5,.04),
               ncol=2, frameon=False, fontsize=9)
    fig.suptitle('Latent-Space MMD² Before and After Adaptation',fontsize=11,fontweight='bold',y=.99)
    fig.text(.5,.012,'Post-hoc latent diagnostic · five training seeds · mean ± sample SD',
             ha='center',fontsize=7,color='#64748B')
    fig.tight_layout(rect=[0,.10,1,.95],h_pad=1.8)
    for ax in axes: fit_category_labels(ax)
    fig.tight_layout(rect=[0,.10,1,.95],h_pad=1.8)
    if EXPORT:
        fig.savefig(OUT/'06_alignment_before_vs_after_mmd.png',dpi=EXPORT_DPI)
    plt.show()
    plt.close(fig)


alignment = load(CANONICAL / "alignment_runs.csv")
latent_mmd_grouped_bar(alignment)
run_plot(alignment, ["domain_auc"], "06 · Alignment before vs after", ("phase",))
mechanism = load(ANALYSIS / "tableau_mechanism/mechanism_summary.csv")
for (family, metric), d in mechanism.groupby(["family", "metric"]):
    summary_plot(
        d,
        f"06 · {label(family)} — {label(metric)}",
        ("method", "phase"),
        ylabel=label(metric),
        note="Post-hoc validation diagnostic · exported 95% CI across five training seeds",
    )
sep = load(ANALYSIS / "tableau_mechanism/class_separation_runs.csv")
run_plot(
    sep,
    ["normal_normal", "attack_attack", "normal_attack", "attack_normal"],
    "06 · Class-pair MMD",
    ("phase",),
)
threshold = load(ANALYSIS / "tableau_mechanism/prediction_threshold_runs.csv")
run_plot(
    threshold,
    [
        "source_threshold",
        "target_ap",
        "roc_auc",
        "recall",
        "fpr",
        "f1_source_threshold",
        "oracle_f1_diagnostic_only",
        "oracle_f1_gap",
    ],
    "06 · Prediction thresholds (oracle is diagnostic only)",
    ("phase",),
)
scores = load(
    ANALYSIS / "tableau_mechanism/prediction_score_quantiles.csv",
    filters={"training_seed": SEED},
)
q = scores[scores.training_seed == SEED]
curves(
    q,
    "quantile_order",
    "score",
    ["phase", "class_group"],
    ["direction", "method"],
    f"06 · Prediction-score quantile profiles — seed {SEED}",
    order="quantile_order",
    xlabel="Score quantile",
    ylabel="Attack score",
    note=f"Target validation · training seed {SEED} · quantile profiles, post-hoc",
)
display(
    q[["quantile_order", "quantile"]].drop_duplicates().sort_values("quantile_order")
)
mmd_gain = load(ANALYSIS / 'tableau_mechanism/mmd_delta_ap_runs.csv')
fig, axes = plt.subplots(1, 2, figsize=(26/2.54, 13/2.54), sharex=True, sharey=True)
methods = ['marginal_mmd', 'mk_mmd', 'class_aware_mmd']
for i, (ax, direction) in enumerate(zip(axes, ['cicids_to_unsw','unsw_to_cicids'])):
    d = mmd_gain[mmd_gain.direction == direction]
    for method in methods:
        g = d[d.method == method]
        ax.scatter(g.delta_mmd2, g.delta_ap, s=50, alpha=.8,
                   label=label(method), color=METHOD_COLORS[method])
    ax.axhline(0, color='#64748B', ls='--', lw=1)
    ax.axvline(0, color='#64748B', ls='--', lw=1)
    ax.set_title(f'({chr(97+i)}) {label(direction)}', fontsize=11, pad=12)
    ax.set_xlabel('ΔMMD² (after − before)', fontsize=10)
    ax.set_ylabel('ΔAP (after − before)', fontsize=10)
    ax.tick_params(labelsize=9)
axes[1].tick_params(labelleft=True)
handles, legend_labels = axes[0].get_legend_handles_labels()
fig.legend(handles, legend_labels, loc='lower center',bbox_to_anchor=(.5,.07),
           ncol=3, frameon=False,fontsize=10)
fig.suptitle('Does lower MMD accompany better target AP?',
             fontsize=14,fontweight='bold',y=.99)
fig.text(.5,.012,'Target validation · one point per method × training seed · association only',
         ha='center',fontsize=8,color='#64748B')
fig.tight_layout(rect=[0,.17,1,.92],w_pad=2)
if EXPORT:
    fig.savefig(OUT/'06_does_lower_mmd_accompany_better_target_ap.png',dpi=EXPORT_DPI)
plt.show()
plt.close(fig)


## 07 · Gradient conflict, source preservation, class prior và BatchNorm shift
Các thước đo cơ chế là post-hoc diagnostics. Hình không chứng minh nguyên nhân của negative transfer.

In [ ]:
diagnostics = load(CANONICAL / "diagnostics_runs.csv")
metrics = [
    "gradient_conflict_mean",
    "gradient_conflict_negative_fraction",
    "source_preservation_source_val_ap_delta",
    "label_prior_source_attack_prevalence",
    "label_prior_target_attack_prevalence",
    "negative_transfer_delta_target_pr_auc",
]
metrics += [
    c
    for c in diagnostics
    if c.startswith("batchnorm_shift_") and ("source_shift" in c or "target_shift" in c)
]
run_plot(diagnostics, metrics, "07 · Post-hoc mechanism diagnostics")

## 08 · Latent kNN — agreement và mixing
Tách comparison, k, query/reference role và class; tránh gộp cross-domain với within-domain. Balanced sample là dữ liệu chẩn đoán, không phản ánh class prevalence thực tế.

In [ ]:
knn = load(ANALYSIS / "latent_knn_balanced_v1/knn_summary.csv")
for (metric, k, comparison, query, reference), d in knn.groupby(
    ["metric", "k", "comparison", "query_role", "reference_role"]
):
    summary_plot(
        d,
        f"08 · kNN {label(metric)} — k{k} — {comparison} — {query} to {reference}",
        ("method", "class_group"),
        ylabel=label(metric),
        note="Balanced validation sample · post-hoc · exported 95% CI across training seeds",
    )
flows = load(ANALYSIS / "latent_knn_balanced_v1/knn_flow_scores.csv")
for (direction, k, comparison, query, reference), d in flows.groupby(
    ["direction", "k", "comparison", "query_role", "reference_role"]
):
    groups = list(d.groupby(["method", "class_group"]))
    fig, ax = plt.subplots(figsize=(FIG_WIDTH, 6.2))
    boxes = ax.boxplot(
        [g.agreement.to_numpy() for _, g in groups],
        tick_labels=[" · ".join((label(x) for x in key)) for key, _ in groups],
        showfliers=False,
        patch_artist=True,
        boxprops={"facecolor": "#DBEAFE", "edgecolor": "#2563EB"},
        medianprops={"color": "#F97316"},
    )
    for patch, ((method, cls), _) in zip(boxes["boxes"], groups):
        patch.set_facecolor(METHOD_COLORS[method])
        patch.set_alpha(0.6)
        if str(cls).lower() == "attack":
            patch.set_hatch("//")
    ax.tick_params(axis="x", rotation=90)
    ax.set_ylabel("Per-flow class agreement")
    finish(
        fig,
        f"08 · kNN flow agreement — {direction} — k{k} — {comparison} — {query} to {reference}",
        "Balanced validation samples · box = IQR · median line · outliers hidden",
    )

## 10 · Figure tổng cho đồ án (A4 ngang)

Các nhóm dưới đây được vẽ lại trực tiếp từ CSV, mỗi panel có trục và đơn vị riêng. Figure tổng rộng **26 cm**, tối đa 24 cm cao, PNG 300 dpi; subfigure (a), (b)… dùng để dẫn chiếu trong bài. Khi dùng trang A4 dọc nên chọn hình đơn 16 cm, tránh thu nhỏ figure tổng khiến chữ khó đọc.

- Hiệu năng: sáu metrics trong một figure, tách từng transfer direction.
- RQ1: sáu so sánh within/cross trên cùng miền đánh giá.
- Feature shift: KS ranking + năm ECDF, cùng một sampling seed cho ECDF.
- Alignment và negative transfer: MMD²/domain AUC/class separation/gradient conflict/ΔAP/source preservation.
- Prediction diagnostics: AP/ROC-AUC/recall/FPR/F1/threshold, giữ phase riêng.
- Latent kNN: agreement/mixing/Δagreement/Δmixing/ΔAP, giữ k và comparison riêng.

CI dùng số có sẵn trong summary. Raw runs hiển thị mean±SD, không gọi SD là CI. Không gộp các export AP khác nhau vào cùng một phép tính.


In [ ]:
def grouped_canvas(n_panels=6):
    cols = 3
    rows = int(np.ceil(n_panels / cols))
    fig, axes = plt.subplots(
        rows, cols, figsize=(26 / 2.54, (10 * rows + 3) / 2.54), squeeze=False
    )
    for ax in list(axes.flat)[n_panels:]:
        ax.set_visible(False)
    return (fig, list(axes.flat)[:n_panels])


def panel_bars(
    ax,
    d,
    metric,
    groups=("method",),
    value="mean",
    low="ci95_low",
    high="ci95_high",
    sd=False,
):
    d = d.reset_index(drop=True)
    x = np.arange(len(d))
    y = d[value].to_numpy(float)
    names = [" · ".join((label(row[c]) for c in groups)) for _, row in d.iterrows()]
    bars = ax.bar(
        x,
        y,
        color=[semantic_color([row[c] for c in groups]) for _, row in d.iterrows()],
        width=0.65,
    )
    bar_style(ax, bars, d)
    if low in d and high in d:
        good = d[[value, low, high]].notna().all(axis=1).to_numpy()
        ax.errorbar(
            x[good],
            y[good],
            yerr=np.maximum(
                0, np.vstack([y[good] - d.loc[good, low], d.loc[good, high] - y[good]])
            ),
            fmt="none",
            ecolor="#334155",
            capsize=3,
        )
    ax.set_xticks(x, names, rotation=90, ha="center")
    ax.set_title(label(metric))
    ax.set_ylabel(label(metric))
    ax.axhline(0, lw=0.7, color="#94A3B8")


def panel_runs(ax, d, metric, groups=("method", "phase")):
    groups = [c for c in groups if c in d]
    stats = d.groupby(groups, dropna=False)[metric].agg(["mean", "std"]).reset_index()
    stats["sd_low"] = stats["mean"] - stats["std"]
    stats["sd_high"] = stats["mean"] + stats["std"]
    panel_bars(ax, stats, metric, tuple(groups), low="sd_low", high="sd_high")


perf = load(CANONICAL / "performance_summary.csv")
for direction, d in perf.groupby("direction"):
    metrics = [
        m
        for m in [
            "pr_auc",
            "roc_auc",
            "macro_f1",
            "recall",
            "fpr",
            "adaptation_gain_pr_auc",
        ]
        if m in d.metric.unique()
    ]
    fig, axes = grouped_canvas(len(metrics))
    for ax, metric in zip(axes, metrics):
        panel_bars(ax, d[d.metric == metric], metric)
    finish(
        fig,
        f"10A · Target performance overview — {label(direction)}",
        "Target validation · mean and exported 95% CI across training seeds",
        large=True,
    )
rq = load(ANALYSIS / "rq1/rq1_summary.csv")
for _, row in rq.iterrows():
    metrics = ["pr_auc", "roc_auc", "macro_f1", "f1_attack", "recall", "fpr"]
    fig, axes = grouped_canvas()
    for ax, metric in zip(axes, metrics):
        d = pd.DataFrame(
            [
                {
                    "evaluation": mode,
                    "mean": row[f"{mode}_{metric}_mean"],
                    "sd_low": row[f"{mode}_{metric}_mean"]
                    - row[f"{mode}_{metric}_std"],
                    "sd_high": row[f"{mode}_{metric}_mean"]
                    + row[f"{mode}_{metric}_std"],
                }
                for mode in ["within", "cross"]
            ]
        )
        panel_bars(ax, d, metric, ("evaluation",), low="sd_low", high="sd_high")
    finish(
        fig,
        f"10B · Within vs cross-domain overview — {label(row.eval_domain)}",
        "Validation · same evaluation domain · mean ± SD across training seeds",
        large=True,
    )
ks = load(ANALYSIS / "tableau_feature_shift_5seeds/ks_summary_common_features.csv")
e = load(
    ANALYSIS / "tableau_feature_shift_5seeds/ecdf_common_features.csv",
    filters={"sampling_seed": SEED},
)
e = e[e.sampling_seed == SEED]
features = ks.feature.tolist()
fig, axes = grouped_canvas(1 + len(features))
panel_bars(
    axes[0],
    ks,
    "KS statistic",
    ("feature",),
    value="ks_mean",
    low="ks_min",
    high="ks_max",
)
axes[0].set_title("KS ranking (mean and min–max)")
for ax, feature in zip(axes[1:], features):
    for domain, d in e[e.feature == feature].groupby("domain"):
        d = d.sort_values("point_order")
        ax.plot(
            d.value_log1p, d.ecdf, label=domain, color=DOMAIN_COLORS[domain], lw=1.5
        )
    ax.set_title(label(feature))
    ax.set_xlabel("Signed log1p(raw value)")
    ax.set_ylabel("ECDF")
    ax.legend(fontsize=8)
finish(
    fig,
    f"10C · Common-feature shift overview — sampling seed {SEED}",
    "KS: all sampling seeds, range is min–max · ECDF: one selected sampling seed · unlabeled training samples",
    large=True,
)
al = load(CANONICAL / "alignment_runs.csv")
sep = load(ANALYSIS / "tableau_mechanism/class_separation_runs.csv")
diag = load(CANONICAL / "diagnostics_runs.csv")
for direction in al.direction.unique():
    fig, axes = grouped_canvas()
    specs = [
        (al, "mmd2", ("method", "phase")),
        (al, "domain_auc", ("method", "phase")),
        (sep, "separation_ratio", ("method", "phase")),
        (diag, "gradient_conflict_mean", ("method",)),
        (diag, "negative_transfer_delta_target_pr_auc", ("method",)),
        (diag, "source_preservation_source_val_ap_delta", ("method",)),
    ]
    for ax, (table, metric, groups) in zip(axes, specs):
        panel_runs(ax, table[table.direction == direction], metric, groups)
    finish(
        fig,
        f"10D · Alignment and negative-transfer overview — {label(direction)}",
        "Post-hoc: validation for alignment/AP, training minibatches for gradients · mean ± SD across seeds",
        large=True,
    )
th = load(ANALYSIS / "tableau_mechanism/prediction_threshold_runs.csv")
for direction, d in th.groupby("direction"):
    fig, axes = grouped_canvas()
    for ax, metric in zip(
        axes,
        [
            "target_ap",
            "roc_auc",
            "recall",
            "fpr",
            "f1_source_threshold",
            "source_threshold",
        ],
    ):
        panel_runs(ax, d, metric)
    finish(
        fig,
        f"10E · Prediction-quality overview — {label(direction)}",
        "Target validation · before/after separated · mean ± SD across training seeds",
        large=True,
    )
kdf = load(ANALYSIS / "latent_knn_balanced_v1/knn_summary.csv")
for (direction, k, comparison, query, reference), d in kdf.groupby(
    ["direction", "k", "comparison", "query_role", "reference_role"]
):
    metrics = [
        m
        for m in ["agreement", "mixing", "delta_agreement", "delta_mixing", "delta_ap"]
        if m in d.metric.unique()
    ]
    fig, axes = grouped_canvas(len(metrics))
    for ax, metric in zip(axes, metrics):
        panel_bars(ax, d[d.metric == metric], metric, ("method", "class_group"))
    finish(
        fig,
        f"10F · Latent kNN overview — {label(direction)} — k{k} — {comparison} — {query} to {reference}",
        "Balanced validation samples · exported mean and 95% CI across training seeds",
        large=True,
    )

## 11 · CSV coverage và hình đã xuất
Các bảng summary/runs phục vụ cùng nhóm hình; metadata/provenance không được biến thành biểu đồ. Snapshot feature shift một seed cũ được thay bằng snapshot 5 seeds, tránh đếm trùng. `score_metrics`, `score_quantiles`, `diagnostic_deltas` có bản chuyên biệt trong `tableau_mechanism`; giữ riêng các export có AP khác nhau do cách tính/độ chính xác để không join nhầm.


In [ ]:
display(pd.DataFrame({"PNG": [p.name for p in sorted(OUT.glob("*.png"))]}))
print(f"Hình được lưu tại: {OUT}")